# 直接波动观测模型：拟合、预测与更新

本单元接收时间正序的一维 RV／BPV，或其已在前处理层完成的 log 变换。支持 ARMA、ARFIMA、HAR、MEM，一次调用只拟合一个模型。采样、分钟贡献、季节性和输入转换由 b03、c01、c02 完成。

预测本序列的下一采样点，保留传入尺度。`require_nonnegative_prediction=True` 时下一步负预测返回失败；False 时保留有限负预测，适用于 log 输入。零预测符合“非负”；MEM 自身要求条件均值严格为正。模型不内部取 log、做整数差分、指数还原或重采样。

## 四类模型

**ARMA／ARFIMA**：$\Phi(L)(1-L)^d(x_t-\mu)=\Theta(L)\epsilon_t$，创新方差为常数。高斯 QML 联合估计中心水平、AR／MA、分数 d 和创新方差。复用 [c03](c03_ReturnModels.ipynb) 的正式均值算法，方差阶数为 (0,0)。AR 平稳、MA 可逆；ARFIMA 的 d 在 (-0.5,0.5)，调用者可指定更窄边界与有限截断。mean.mu 是中心水平；innovation.variance 是波动观测的加性创新方差，不是 RV／BPV 预测本身。

**HAR**：
$$\widehat x_t=b_0+\sum_{w\in W}b_w\frac1w\sum_{j=1}^w x_{t-j}.$$
窗口按采样点数，均截止于上一已实现点，可重叠，窗口及数量由调用者选择。截距和窗口系数由 NNLS 非负最小二乘估计，不限制系数和，不要求平稳。include_mean=False 时不估计截距。log 输入的窗口均值就是传入 log 值的均值。

**MEM**：
$$x_t=\mu_t e_t,\quad\mu_t=\omega+\sum_i\alpha_i x_{t-i}+\sum_j\beta_j\mu_{t-j}.$$
输入非负，允许零；$\omega>0$，$\alpha_i,\beta_j\ge0$，$\sum\alpha+\sum\beta<1$。基础单序列形式使用准似然目标 $\sum_t[\log\mu_t+x_t/\mu_t]$，每次候选参数重新计算全训练路径。输出乘性误差 $x_t/\mu_t$，零观测对应零误差；不增加创新分布选项。定义与准似然性质见[作者论文](https://igier.unibocconi.eu/sites/default/files/media/attach/GalloPAPER.pdf)，NNLS 见 [SciPy 文档](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.nnls.html)。


## 显式运行准备

根定位仅用于直接运行本 Notebook；代理只加载 export 定义，调用方负责模块路径。

In [31]:
from pathlib import Path
import sys
from IPython.display import display

project_markers = [".git", ".env", "config/settings.py"]
current_path = Path.cwd().resolve()
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        if str(candidate_root) not in sys.path:
            sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")
from config.settings import settings
if str(candidate_root / "04_Research") not in sys.path:
    sys.path.insert(0, str(candidate_root / "04_Research"))


In [32]:
from __future__ import annotations

from copy import deepcopy
import numpy as np
import pandas as pd
from scipy.optimize import minimize, nnls
from scipy.special import softmax
from numba import njit
from a01_Methods.b04_ReturnVolatilityModeling import c03_ReturnModels as return_models

VOLATILITY_MODEL_NAMES = ("ARMA", "ARFIMA", "HAR", "MEM")


In [33]:
def har_window_features(observations, *, windows, presample=None) -> pd.DataFrame:
    """每行仅使用该行之前的最近 W 个采样观测；不填不足历史。"""
    windows = tuple(windows)
    if not windows or any(not isinstance(w, (int, np.integer)) or isinstance(w, (bool, np.bool_)) or w < 1 for w in windows):
        raise ValueError("windows 必须为非空正整数窗口序列")
    if len(set(windows)) != len(windows):
        raise ValueError("windows 不可重复，具名系数须与窗口一一对应")
    values = (observations.to_numpy(dtype=float, na_value=np.nan) if isinstance(observations, pd.Series)
              else np.asarray(observations, dtype=float))
    previous = np.asarray([] if presample is None else presample, dtype=float)
    if values.ndim != 1 or previous.ndim != 1 or not np.isfinite(values).all() or not np.isfinite(previous).all():
        raise ValueError("HAR 观测及 presample 必须为有限一维序列，不删除缺失行")
    previous = previous[-max(windows):]
    combined = pd.Series(np.r_[previous, values])
    features_df = pd.DataFrame({f"window_mean[{w}]": combined.shift(1).rolling(w, min_periods=w).mean().iloc[len(previous):].to_numpy()
                               for w in windows}, index=observations.index.copy() if isinstance(observations, pd.Series) else pd.RangeIndex(len(values)))
    for width in windows:
        complete = np.arange(len(values)) + len(previous) >= width
        if not np.isfinite(features_df[f"window_mean[{width}]"].to_numpy()[complete]).all():
            raise ValueError("完整 HAR 窗口均值非有限；不能按历史不足删除该行")
    return features_df


@njit(cache=False)
def _filter_mem_sequence(observations, omega, alpha, beta, initial_mean, past_observations, past_means):
    """拟合与逐点更新共用的 MEM 条件均值递推；无正数裁剪。"""
    means = np.full(len(observations), np.nan)
    for position in range(len(observations)):
        conditional_mean = omega
        for lag in range(1, len(alpha) + 1):
            source = position - lag
            if source >= 0:
                previous = observations[source]
            elif len(past_observations) + source >= 0:
                previous = past_observations[len(past_observations) + source]
            else:
                previous = initial_mean
            conditional_mean += alpha[lag - 1] * previous
        for lag in range(1, len(beta) + 1):
            source = position - lag
            if source >= 0:
                previous = means[source]
            elif len(past_means) + source >= 0:
                previous = past_means[len(past_means) + source]
            else:
                previous = initial_mean
            conditional_mean += beta[lag - 1] * previous
        if not np.isfinite(conditional_mean) or conditional_mean <= 0:
            return means
        means[position] = conditional_mean
    return means


def _decode_mem_parameters(raw_parameters, orders):
    observation_order, mean_order = orders
    weights = softmax(np.r_[raw_parameters[1:], 0.0])[:-1]
    return float(np.exp(raw_parameters[0])), weights[:observation_order], weights[observation_order:observation_order + mean_order]


## 独立调用与初始化

`har_window_features(observations, windows=..., presample=...)` 可独立计算 HAR 特征。第 i 行只用 i 之前的观测，不足完整窗口保留 NaN。该函数不拟合模型。

`fit_volatility_model(observations, model=..., ...)` 返回 status、reason、parameters、specification、in_sample、optimization、loglikelihood、state、next_forecast。Pandas Series 保留索引；数组使用顺序索引。

`predict_volatility_model(state)` 返回下一步 mean_forecast，不修改状态。HAR 还返回窗口均值 features 和截距／窗口贡献 components；ARMA／ARFIMA 还返回创新方差 innovation_variance。`update_volatility_model(state, observed_value)` 吸收一个实现值，返回新 state 和下一步预测，参数不变且旧状态不推进。

ARMA／ARFIMA 指定 mean_order=(AR,MA)，ARFIMA 显式指定 fractional_memory，可限制 fractional_d_bounds。初始化沿用 c03 的 backcast／sample_variance，initial_variance 可覆盖初始创新方差；presample 的 observations、innovations 对应前历史。

HAR 指定 har_windows=(...)。可用 presample.observations 提供紧邻训练期之前的已知观测；不足历史时训练开头仅用于滞后构造，in_sample 保留这些行及 insufficient_window_history 状态。NNLS 使用完整窗口后的训练行，optimization 记录实际拟合行数和起始历史行数。要求可用行数超过自由参数数，设计矩阵满秩。不删除内部缺失。系数和 ≥1 不判失败。

MEM 指定 mem_order=(观测滞后阶数,条件均值滞后阶数)，各为非负整数。initial_mean 可指定正初始均值；否则 backcast 使用训练前 min(n,75) 行、0.94 几何权重的加权均值，sample_mean 使用训练均值。presample.observations 非负，presample.means 为正；未提供前历史以初始均值代入。不接受 sample_variance 或 include_mean=False。

numeric_scale 仅用于优化的线性换单位：ARMA／ARFIMA、HAR 默认训练标准差，MEM 默认训练均值。系数、状态、特征、成分及预测均回到输入尺度。HAR 优化选项仅支持 maxiter，其余可传 L-BFGS-B 选项。

每次只调用一次拟合优化，不选阶、重试、换模型或填值。配置无效、样本不足、内部缺失、不可识别、不收敛或无法有效预测时返回 failed 和原因，state 为 None。非负要求检查返回的下一步预测，不修改训练目标或强行约束 ARMA 系数。失败更新不推进旧状态；成功状态只保留有限历史。loglikelihood 在 ARMA／ARFIMA 为 Gaussian QML 值，在 MEM 为准似然的负和，在 HAR 为 None，不用于本单元跨模型排名。


In [34]:
def fit_volatility_model(
    observations, *, model: str, mean_order: tuple[int, int] | None = None,
    har_windows=None, mem_order: tuple[int, int] | None = None,
    include_mean: bool = True, fractional_memory: int | None = None,
    fractional_d_bounds: tuple[float, float] | None = None,
    initialization: str = "backcast", initial_mean: float | None = None,
    initial_variance: float | None = None, presample: dict | None = None,
    numeric_scale: float | None = None, require_nonnegative_prediction: bool = False,
    optimizer_options: dict | None = None,
) -> dict:
    """一次拟合一个直接波动观测模型，输入变换和调度由调用者负责。"""
    failure = {"status": "failed", "reason": "", "parameters": {}, "state": None}
    try:
        if model not in VOLATILITY_MODEL_NAMES:
            raise ValueError("model 必须为 ARMA、ARFIMA、HAR 或 MEM")
        if not isinstance(include_mean, (bool, np.bool_)) or not isinstance(require_nonnegative_prediction, (bool, np.bool_)):
            raise ValueError("include_mean 与 require_nonnegative_prediction 必须为布尔值")
        if initialization not in ("backcast", "sample_mean", "sample_variance"):
            raise ValueError("initialization 不属于可用初始化方式")
        if optimizer_options is not None and not isinstance(optimizer_options, dict):
            raise ValueError("optimizer_options 必须为字典")
        if isinstance(observations, pd.Series):
            values = observations.to_numpy(dtype=float, na_value=np.nan)
            observation_index = observations.index.copy()
        else:
            values = np.asarray(observations, dtype=float)
            observation_index = pd.RangeIndex(len(values)) if values.ndim else pd.RangeIndex(0)
        if values.ndim != 1 or not len(values) or not np.isfinite(values).all():
            raise ValueError("observations 必须为非空有限一维序列；不自动删除缺失行")
        history = {} if presample is None else presample
        if not isinstance(history, dict):
            raise ValueError("presample 必须为字典")
        allowed_history = ({"observations", "innovations"} if model in ("ARMA", "ARFIMA") else
                           {"observations"} if model == "HAR" else {"observations", "means"})
        if set(history) - allowed_history:
            raise ValueError(f"{model} 的 presample 仅使用 {sorted(allowed_history)}")
        for name, past in history.items():
            past = np.asarray(past, dtype=float)
            if past.ndim != 1 or not np.isfinite(past).all():
                raise ValueError(f"presample.{name} 必须为有限一维历史")
        specification = {"model": model, "include_mean": bool(include_mean),
            "require_nonnegative_prediction": bool(require_nonnegative_prediction), "initialization": initialization}
        if model in ("ARMA", "ARFIMA"):
            if mean_order is None or har_windows is not None or mem_order is not None or initial_mean is not None:
                raise ValueError("ARMA／ARFIMA 必须指定 mean_order，不使用 HAR／MEM 配置")
            if initialization == "sample_mean":
                raise ValueError("ARMA／ARFIMA 的初始化使用 backcast 或 sample_variance")
            # 复用正式均值算法，(0,0) 方差阶数表示常方差 Gaussian QML。
            mean_fit = return_models.fit_return_model(
                pd.Series(values, index=observation_index), model="ARFIMA-GARCH" if model == "ARFIMA" else "ARMA-GARCH",
                mean_order=mean_order, volatility_order=(0, 0), include_mean=include_mean,
                fractional_memory=fractional_memory, fractional_d_bounds=fractional_d_bounds,
                initialization=initialization, initial_variance=initial_variance,
                presample={"returns": history.get("observations", []), "residuals": history.get("innovations", [])},
                numeric_scale=numeric_scale, optimizer_options=optimizer_options)
            if mean_fit["status"] != "ok":
                return {**failure, "reason": mean_fit["reason"], "optimization": mean_fit.get("optimization"), "specification": specification}
            specification.update({k: mean_fit["specification"][k] for k in
                                  ("mean_order", "fractional_memory", "fractional_d_bounds", "numeric_scale", "estimation")})
            parameters = {("innovation.variance" if name == "variance.omega" else name): value
                          for name, value in mean_fit["parameters"].items()}
            state = {"status": "ok", "specification": specification, "parameters": parameters,
                     "mean_state": mean_fit["state"], "observations_seen": len(values)}
            in_sample_df = mean_fit["in_sample"].rename(columns={"conditional_variance": "innovation_variance"})
            optimization, loglikelihood = mean_fit["optimization"], mean_fit["loglikelihood"]
        else:
            if mean_order is not None or fractional_memory is not None or fractional_d_bounds is not None or initial_variance is not None:
                raise ValueError("HAR／MEM 不使用 ARMA、分数滤波或初始创新方差配置")
            if model == "HAR":
                if har_windows is None or mem_order is not None or initial_mean is not None:
                    raise ValueError("HAR 必须指定 har_windows，不使用 MEM 配置或历史填充均值")
                windows = tuple(har_windows)
                features_df = har_window_features(pd.Series(values, index=observation_index), windows=windows,
                                                   presample=history.get("observations"))
                valid = np.isfinite(features_df.to_numpy()).all(axis=1)
                free_parameters = len(windows) + int(include_mean)
                if valid.sum() <= free_parameters:
                    raise ValueError("完整 HAR 窗口后的训练行数须超过自由参数数")
                scale = float(np.std(values[valid])) if numeric_scale is None else float(numeric_scale)
                if not np.isfinite(scale) or scale <= 0:
                    raise ValueError("HAR numeric_scale 须为有限正数；训练目标须有有效变化")
                design = features_df.to_numpy()[valid] / scale
                if include_mean:
                    design = np.column_stack((np.ones(valid.sum()), design))
                if not np.isfinite(design).all() or np.linalg.matrix_rank(design) < free_parameters:
                    raise ValueError("HAR 设计矩阵非有限或秩不足，具名系数无法识别")
                options = {} if optimizer_options is None else optimizer_options.copy()
                if set(options) - {"maxiter"}:
                    raise ValueError("HAR optimizer_options 仅支持 NNLS 的 maxiter")
                coefficients, residual_norm = nnls(design, values[valid] / scale, **options)
                if not np.isfinite(coefficients).all() or not np.isfinite(residual_norm):
                    raise ValueError("HAR NNLS 系数或目标非有限")
                intercept = float(coefficients[0] * scale) if include_mean else 0.0
                window_coefficients = coefficients[int(include_mean):].copy()
                parameters = ({"mean.intercept": intercept} if include_mean else {})
                parameters.update({f"mean.window[{w}]": float(coef) for w, coef in zip(windows, window_coefficients)})
                fitted = np.full(len(values), np.nan)
                fitted[valid] = intercept + features_df.to_numpy()[valid] @ window_coefficients
                innovations = values - fitted
                if not np.isfinite(fitted[valid]).all() or not np.isfinite(innovations[valid]).all():
                    raise ValueError("HAR 原尺度拟合路径或创新非有限")
                specification.update({"har_windows": tuple(map(int, windows)), "numeric_scale": scale,
                                      "estimation": "nnls", "stationarity_required": False})
                state = {"status": "ok", "specification": specification, "parameters": parameters,
                    "intercept": intercept, "window_coefficients": window_coefficients,
                    "observations_history": np.r_[history.get("observations", []), values][-max(windows):].copy(),
                    "history_limit": max(windows), "observations_seen": len(values)}
                in_sample_df = pd.DataFrame({"observation": values, "conditional_mean": fitted,
                    "innovation": innovations, "fit_status": np.where(valid, "ok", "insufficient_window_history")}, index=observation_index)
                in_sample_df = pd.concat((in_sample_df, features_df), axis=1)
                optimization = {"method": "NNLS", "success": True, "iterations": None,
                    "message": "非负截距与窗口系数；不限制系数和", "training_rows": int(valid.sum()),
                    "initial_history_rows": int((~valid).sum()), "free_parameters": free_parameters,
                    "scaled_residual_sum_squares": float(residual_norm ** 2)}
                loglikelihood = None
            else:
                if mem_order is None or har_windows is not None or not include_mean:
                    raise ValueError("MEM 必须指定 mem_order，保持正截距，不使用 HAR 配置")
                if len(mem_order) != 2 or any(not isinstance(v, (int, np.integer)) or isinstance(v, (bool, np.bool_)) or v < 0 for v in mem_order):
                    raise ValueError("mem_order 必须为两个非负整数：观测滞后、均值滞后")
                orders = tuple(map(int, mem_order))
                if initialization == "sample_variance":
                    raise ValueError("MEM 的均值初始化使用 backcast 或 sample_mean")
                if np.any(values < 0):
                    raise ValueError("MEM 输入必须非负，包括前处理后的 log 输入")
                average = float(np.mean(values))
                scale = average if numeric_scale is None else float(numeric_scale)
                if not np.isfinite(average) or average <= 0 or not np.isfinite(scale) or scale <= 0:
                    raise ValueError("MEM 训练均值和 numeric_scale 必须严格为正且有限")
                scaled_values = values / scale
                if initial_mean is None:
                    length = min(len(values), 75)
                    weights = .94 ** np.arange(length)
                    seed_mean = float(weights @ scaled_values[:length] / weights.sum()) if initialization == "backcast" else float(np.mean(scaled_values))
                else:
                    seed_mean = float(initial_mean) / scale
                if not np.isfinite(seed_mean) or seed_mean <= 0:
                    raise ValueError("MEM 初始均值必须严格为正且有限")
                past_observations = np.asarray(history.get("observations", []), dtype=float)
                past_means = np.asarray(history.get("means", []), dtype=float)
                if np.any(past_observations < 0) or np.any(past_means <= 0):
                    raise ValueError("MEM 历史观测须非负，历史条件均值须为正")
                raw_weights = np.r_[np.full(orders[0], .10 / orders[0]) if orders[0] else [],
                                     np.full(orders[1], .80 / orders[1]) if orders[1] else []]
                raw_start = np.r_[np.log(np.mean(scaled_values) * (1 - raw_weights.sum())),
                                  np.log(raw_weights / (1 - raw_weights.sum()))]
                if len(values) <= max(len(raw_start), *orders):
                    raise ValueError("MEM 训练观测数须超过自由参数数与各阶数")
                def objective(raw_parameters):
                    omega, alpha, beta = _decode_mem_parameters(raw_parameters, orders)
                    means = _filter_mem_sequence(scaled_values, omega, alpha, beta, seed_mean,
                                                 past_observations / scale, past_means / scale)
                    with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
                        score = np.mean(np.log(means) + scaled_values / means)
                    return float(score) if np.isfinite(score) else 1.e12
                options = {"maxiter": 500, "maxfun": 100000, "ftol": 1.e-10, "gtol": 1.e-5, "maxls": 40}
                if optimizer_options is not None:
                    options.update(optimizer_options)
                initial_score = objective(raw_start)
                optimum = minimize(objective, raw_start, method="L-BFGS-B",
                                   bounds=[(-30., 30.)] + [(-20., 20.)] * (len(raw_start) - 1), options=options)
                optimization = {"method": "L-BFGS-B", "success": bool(optimum.success),
                    "message": str(optimum.message), "iterations": int(optimum.nit), "evaluations": int(optimum.nfev),
                    "initial_average_qml": initial_score, "final_average_qml": float(optimum.fun),
                    "free_parameters": len(raw_start), "training_rows": len(values)}
                if not optimum.success:
                    return {**failure, "reason": f"MEM 优化未收敛: {optimum.message}", "optimization": optimization,
                            "specification": specification}
                omega, alpha, beta = _decode_mem_parameters(optimum.x, orders)
                omega *= scale
                means = _filter_mem_sequence(values, omega, alpha, beta, seed_mean * scale, past_observations, past_means)
                with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
                    multiplicative_errors = values / means
                    innovations = values - means
                    loglikelihood = -float(np.sum(np.log(means) + multiplicative_errors))
                if (not np.isfinite(means).all() or np.any(means <= 0) or not np.isfinite(multiplicative_errors).all()
                    or not np.isfinite(innovations).all() or not np.isfinite(loglikelihood)
                    or alpha.sum() + beta.sum() >= 1 or omega <= 0):
                    raise ValueError("MEM 原尺度路径、准似然或系数有效性失败")
                parameters = {"mean.omega": float(omega)}
                parameters.update({f"mean.alpha[{i}]": float(v) for i, v in enumerate(alpha, 1)})
                parameters.update({f"mean.beta[{i}]": float(v) for i, v in enumerate(beta, 1)})
                specification.update({"mem_order": orders, "numeric_scale": scale, "estimation": "mem_qml"})
                limit = max(*orders, 1)
                state = {"status": "ok", "specification": specification, "parameters": parameters,
                    "omega": float(omega), "alpha": alpha, "beta": beta, "initial_mean": seed_mean * scale,
                    "observations_history": np.r_[past_observations, values][-limit:].copy(),
                    "means_history": np.r_[past_means, means][-limit:].copy(),
                    "history_limit": limit, "observations_seen": len(values)}
                in_sample_df = pd.DataFrame({"observation": values, "conditional_mean": means,
                    "innovation": innovations, "multiplicative_error": multiplicative_errors}, index=observation_index)
        forecast = predict_volatility_model(state)
        if forecast["status"] != "ok":
            return {**failure, "reason": forecast["reason"], "specification": specification, "optimization": optimization}
        return {"status": "ok", "reason": "", "parameters": parameters.copy(),
            "specification": specification.copy(), "state": state, "in_sample": in_sample_df,
            "optimization": optimization, "loglikelihood": loglikelihood, "next_forecast": forecast}
    except (ValueError, TypeError, RuntimeError, FloatingPointError, OverflowError, np.linalg.LinAlgError) as error:
        return {**failure, "reason": str(error)}


def predict_volatility_model(state: dict) -> dict:
    """预测下一采样观测，不推进 state，不改变尺度或裁剪。"""
    if not isinstance(state, dict) or state.get("status") != "ok":
        return {"status": "failed", "reason": "预测需要成功拟合或更新后的有效 state"}
    model = state["specification"]["model"]
    output = {"status": "ok", "reason": ""}
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        if model in ("ARMA", "ARFIMA"):
            forecast = return_models.predict_return_model(state["mean_state"])
            if forecast["status"] != "ok":
                return {"status": "failed", "reason": forecast["reason"]}
            mean = forecast["mean_forecast"]
            output["innovation_variance"] = forecast["variance_forecast"]
        elif model == "HAR":
            windows = state["specification"]["har_windows"]
            previous = state["observations_history"]
            if len(previous) < max(windows):
                return {"status": "failed", "reason": "HAR 历史不足以形成完整窗口"}
            features = {f"window_mean[{w}]": float(np.mean(previous[-w:])) for w in windows}
            components = {"intercept": state["intercept"]}
            components.update({f"window[{w}]": float(coef * features[f"window_mean[{w}]"])
                               for w, coef in zip(windows, state["window_coefficients"])})
            mean = float(sum(components.values()))
            output.update({"features": features, "components": components})
        elif model == "MEM":
            mean = float(_filter_mem_sequence(np.array([0.0]), state["omega"], state["alpha"], state["beta"],
                         state["initial_mean"], state["observations_history"], state["means_history"])[0])
        else:
            return {"status": "failed", "reason": "state 不属于直接波动观测模型"}
    if not np.isfinite(mean):
        return {"status": "failed", "reason": "下一步预测非有限"}
    if model == "MEM" and mean <= 0:
        return {"status": "failed", "reason": "MEM 下一步条件均值必须为正"}
    if state["specification"]["require_nonnegative_prediction"] and mean < 0:
        return {"status": "failed", "reason": "下一步预测为负，违反调用者的非负预测要求"}
    return {**output, "mean_forecast": mean}


def update_volatility_model(state: dict, observed_value: float) -> dict:
    """吸收一个实现值，返回参数固定的新状态；失败不推进旧状态。"""
    if not isinstance(state, dict) or state.get("status") != "ok":
        return {"status": "failed", "reason": "更新需要成功拟合或更新后的有效 state", "state": None}
    try:
        observed_value = float(observed_value)
    except (ValueError, TypeError, OverflowError):
        return {"status": "failed", "reason": "实现观测必须为一个有限数值", "state": None}
    if not np.isfinite(observed_value):
        return {"status": "failed", "reason": "实现观测缺失或非有限；旧状态未推进", "state": None}
    model = state["specification"]["model"]
    if model == "MEM" and observed_value < 0:
        return {"status": "failed", "reason": "MEM 实现观测必须非负；旧状态未推进", "state": None}
    forecast = predict_volatility_model(state)
    if forecast["status"] != "ok":
        return {"status": "failed", "reason": forecast["reason"], "state": None}
    updated_state = deepcopy(state)
    output = {"status": "ok", "reason": "", "conditional_mean": forecast["mean_forecast"]}
    if model in ("ARMA", "ARFIMA"):
        update = return_models.update_return_model(state["mean_state"], observed_value)
        if update["status"] != "ok":
            return {"status": "failed", "reason": update["reason"], "state": None}
        updated_state["mean_state"] = update["state"]
        output.update({"innovation": update["innovation"], "innovation_variance": update["conditional_variance"],
                       "standardized_innovation": update["standardized_innovation"]})
    else:
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            innovation = float(observed_value - forecast["mean_forecast"])
            output["innovation"] = innovation
            if model == "MEM":
                output["multiplicative_error"] = float(observed_value / forecast["mean_forecast"])
                updated_state["means_history"] = np.r_[state["means_history"], forecast["mean_forecast"]][-state["history_limit"]:]
            else:
                output.update({"features": forecast["features"], "components": forecast["components"]})
        if not np.isfinite(innovation) or (model == "MEM" and not np.isfinite(output["multiplicative_error"])):
            return {"status": "failed", "reason": "实现创新或乘性误差非有限；旧状态未推进", "state": None}
        updated_state["observations_history"] = np.r_[state["observations_history"], observed_value][-state["history_limit"]:]
    updated_state["observations_seen"] += 1
    next_forecast = predict_volatility_model(updated_state)
    if next_forecast["status"] != "ok":
        return {"status": "failed", "reason": f"吸收观测后下一步无法有效预测: {next_forecast['reason']}", "state": None}
    return {**output, "state": updated_state, "next_forecast": next_forecast}


In [35]:
VOLATILITY_MODEL_FIELD_LABELS = {
    "model": "模型", "status": "状态", "reason": "原因", "mean_forecast": "下一步观测预测",
    "observation": "实现观测", "conditional_mean": "条件均值", "innovation": "加性创新",
    "innovation_variance": "观测创新方差", "standardized_innovation": "标准化创新",
    "multiplicative_error": "乘性误差", "fit_status": "训练行状态",
    "parameter": "具名系数", "value": "系数值", "iterations": "优化迭代数",
}
VOLATILITY_MODEL_COLUMN_LABELS = {name: f"{name}（{label}）" for name, label in VOLATILITY_MODEL_FIELD_LABELS.items()}


In [36]:
import pyarrow as pa
from copy import deepcopy
from a01_Methods.b04_ReturnVolatilityModeling.c02_ModelObservations import transform_model_input

demo_prediction_fields = [
    pa.field("sample_id", pa.int64(), nullable=False), pa.field("sample_at", pa.timestamp("us", tz="Asia/Shanghai"), nullable=False),
    pa.field("trading_date", pa.date32(), nullable=False), pa.field("sample_side", pa.string(), nullable=False),
    pa.field("training_start_sample_id", pa.int64()), pa.field("training_end_sample_id", pa.int64()),
    pa.field("training_points", pa.int64(), nullable=False), pa.field("model", pa.string(), nullable=False),
    pa.field("status", pa.string(), nullable=False), pa.field("reason", pa.string(), nullable=False),
    pa.field("mean_forecast", pa.float64()), pa.field("parameters", pa.map_(pa.string(), pa.float64())),
    pa.field("update_status", pa.string()), pa.field("update_reason", pa.string()),
    pa.field("updated_mean_forecast", pa.float64()),
]

def _compute_volatility_demo(demo_observations, *, history_points=500,
                             cases=None, target_positions=None):
    """RV／BPV 的四类逐点真实调用；分片保留完整历史，不保存数据文件。"""
    demo_history_points = history_points
    selected_positions = np.arange(len(demo_observations), dtype=int) if target_positions is None else np.asarray(target_positions, dtype=int)
    if selected_positions.ndim != 1 or len(np.unique(selected_positions)) != len(selected_positions):
        raise ValueError("目标坐标必须是一维不重复位置")
    if len(selected_positions) and (selected_positions.min() < 0 or selected_positions.max() >= len(demo_observations)):
        raise ValueError("目标坐标超出完整输入范围")
    assert demo_observations.iloc[0]["log_return_status"] == "no_previous_sample"
    demo_options = {
        "ARMA": {"mean_order": (1, 1)},
        "ARFIMA": {"mean_order": (1, 1), "fractional_memory": 30},
        "HAR": {"har_windows": (1, 5, 20)},
        "MEM": {"mem_order": (1, 1)},
    }
    demo_result_schema = pa.schema([*demo_prediction_fields,
        pa.field("innovation_variance", pa.float64()),
        pa.field("features", pa.map_(pa.string(), pa.float64())), pa.field("components", pa.map_(pa.string(), pa.float64()))])
    demo_results = {}
    demo_update_checks = {}
    for measure in ("rv", "bpv"):
        if cases is not None and not any(m == measure for m, _ in cases):
            continue
        inputs = transform_model_input(demo_observations, observation_col=measure, require_nonnegative=True)
        values = inputs["model_input"].to_numpy(dtype=float, na_value=np.nan)
        # 特征函数接受有限序列；按原坐标连续有效段调用，不跨缺失拼接。
        # 以下逐点 fit 仍使用完整 values 数组，缺失留在原训练窗口并返回失败。
        source = values[1:]
        finite = np.isfinite(source)
        edges = np.diff(np.r_[False, finite, False].astype(int))
        for left, right in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)):
            segment = source[left:right]
            features = har_window_features(pd.Series(segment), windows=demo_options["HAR"]["har_windows"])
            for width in demo_options["HAR"]["har_windows"]:
                expected = np.full(len(segment), np.nan)
                expected[width:] = [np.mean(segment[i - width:i]) for i in range(width, len(segment))]
                np.testing.assert_allclose(features[f"window_mean[{width}]"], expected, rtol=1e-9, atol=1e-12, equal_nan=True)
        for model_name, options in demo_options.items():
            if cases is not None and (measure, model_name) not in cases:
                continue
            rows, update_checks = [], 0
            for completed, position in enumerate(selected_positions, 1):
                target = inputs.iloc[position]
                start = max(1, position - demo_history_points)
                history = values[start:position]
                row = dict(sample_id=target.sample_id, sample_at=target.sample_at, trading_date=target.trading_date,
                    sample_side=target.sample_side, training_start_sample_id=inputs.iloc[start].sample_id if len(history) else None,
                    training_end_sample_id=inputs.iloc[position - 1].sample_id if len(history) else None,
                    training_points=len(history), model=model_name, parameters=None, features=None, components=None)
                fit = fit_volatility_model(history, model=model_name, require_nonnegative_prediction=True, **options)
                row.update(status=fit["status"], reason=fit["reason"])
                if fit["status"] == "ok":
                    original_state = deepcopy(fit["state"])
                    prediction = predict_volatility_model(fit["state"])
                    row.update(status=prediction["status"], reason=prediction["reason"], parameters=fit["parameters"])
                    if prediction["status"] == "ok":
                        row.update({k: v for k, v in prediction.items() if k not in ("status", "reason")})
                        np.testing.assert_allclose(prediction["mean_forecast"], fit["next_forecast"]["mean_forecast"])
                        update = update_volatility_model(fit["state"], target.model_input)
                        row.update(update_status=update["status"], update_reason=update["reason"])
                        assert fit["state"]["observations_seen"] == original_state["observations_seen"]
                        if "observations_history" in fit["state"]:
                            np.testing.assert_array_equal(fit["state"]["observations_history"], original_state["observations_history"])
                        if update["status"] == "ok":
                            assert update["state"]["parameters"] == fit["parameters"]
                            assert update["state"]["observations_seen"] == len(history) + 1
                            np.testing.assert_allclose(update["conditional_mean"], prediction["mean_forecast"])
                            row["updated_mean_forecast"] = update["next_forecast"]["mean_forecast"]
                            update_checks += 1
                        if model_name == "HAR":
                            np.testing.assert_allclose(prediction["mean_forecast"], sum(prediction["components"].values()))
                rows.append(row)
                if completed % 100 == 0 or completed == len(selected_positions):
                    print(f"c04 {measure} {model_name} {completed}/{len(selected_positions)} 点；成功更新检查 {update_checks}", flush=True)
            result = pd.DataFrame(rows).reindex(columns=demo_result_schema.names)
            assert len(result) == len(selected_positions)
            assert result.sample_id.tolist() == inputs.iloc[selected_positions].sample_id.tolist()
            assert result.loc[result.training_end_sample_id.notna(), "training_end_sample_id"].lt(
                result.loc[result.training_end_sample_id.notna(), "sample_id"]).all()
            demo_results[(measure, model_name)] = result
            demo_update_checks[(measure, model_name)] = update_checks
    return dict(results=demo_results, update_checks=demo_update_checks, schema=demo_result_schema)


## 真实行情内存试算与完整云端 demo

内存试算的示例历史窗口为 30 点，完整云 demo 为 500 点。下面使用 RB 的 2024-12-31—2025-01-02 真实复权分钟行情，在 Notebook 内试算全部函数，仅展示结果，不写数据文件。完整 demo 使用上游 2010-01-01—2026-09-30 的全部可用序列，由云端调用下列同一套组合函数；每个目标点均用此前最多 500 个采样观测重新拟合，失败位置与原因保留。季节窗口 1 年／3 年、N=15、GK sigma×全合约成交额轴等是方法 demo 示例参数，正式大实验配置另行定义。


In [37]:
from pathlib import Path
import sys
from IPython.display import display

project_markers = [".git", ".env", "config/settings.py"]
current_path = Path.cwd().resolve()
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        sys.path.insert(0, str(candidate_root / "04_Research"))
        break
else:
    raise RuntimeError("未找到项目根目录")
from config.settings import settings
from config.data_contracts import FUTURES_MINUTE_SCHEMA

# 本地仅做真实小范围内存试算，不写 demo 数据文件。
demo_variety = "RB"
demo_source_range = "train20100101-20241231_test20250101-20260930"
demo_start_date = pd.Timestamp("2024-12-31")
demo_end_date = pd.Timestamp("2025-01-02")
demo_test_start = pd.Timestamp("2025-01-01")
demo_sampling = "N15_gk_money_all_contracts_log_interpolate_fixed"
demo_base = candidate_root / "04_Research/a01_Methods"
demo_modeling = demo_base / "b04_ReturnVolatilityModeling"
demo_window_names = ("1y", "3y")
demo_history_points = 30  # 小范围内存试算；完整云 demo 固定使用 500 点。
# 完整 demo 由显式云入口执行；此范围不改变正式大实验截至 2026-06-30 的 scope。
demo_cloud_scope = {"start": "2010-01-01", "end": "2026-09-30", "test_start": "2025-01-01"}


In [38]:
from a01_Methods.b04_ReturnVolatilityModeling.c01_IntradaySeasonality import (
    _load_intraday_demo_inputs, _compute_intraday_demo)
from a01_Methods.b04_ReturnVolatilityModeling.c02_ModelObservations import _compute_observations_demo, _load_demo_sampling_axis
if "_b04_observations_preview" not in globals():
    if "_b04_intraday_preview" not in globals():
        demo_real_inputs = _load_intraday_demo_inputs(candidate_root, start=demo_start_date,
            end=demo_end_date, test_start=demo_test_start)
        _b04_intraday_preview = _compute_intraday_demo(demo_real_inputs["main"], demo_real_inputs["calendar"],
            start=demo_start_date, test_start=demo_test_start)
    demo_sampling_axis = _load_demo_sampling_axis(candidate_root, start=demo_start_date, end=demo_end_date)
    _b04_observations_preview = _compute_observations_demo(_b04_intraday_preview["contributions"],
        _b04_intraday_preview["seasonality"], test_start=demo_test_start, sampling_axis=demo_sampling_axis)
demo_observations = _b04_observations_preview["observation_sets"]["none"]


In [39]:
demo_prediction_labels = {"sample_id": "sample_id（采样编号）", "sample_at": "sample_at（预测目标时刻）",
    "trading_date": "trading_date（交易日）", "sample_side": "sample_side（采样侧）",
    "training_start_sample_id": "training_start_sample_id（训练首个编号）",
    "training_end_sample_id": "training_end_sample_id（训练末个编号）", "training_points": "training_points（训练观测数）",
    "parameters": "parameters（具名拟合系数）", "update_status": "update_status（更新状态）",
    "update_reason": "update_reason（更新原因）", "updated_mean_forecast": "updated_mean_forecast（更新后下一步预测）",
    "features": "features（HAR 窗口均值）", "components": "components（预测成分）"}


每个目标使用完整输入中此前最多 500 个采样观测重新拟合；初始窗口未满时使用已有历史，首个锚点及拟合失败也保留。云端节点只分配目标坐标，不将每个节点的稀疏点集作为训练序列。Notebook 仅试算所选小区间，完整上游区间由云端验收。


In [40]:
demo_prediction_preview = _compute_volatility_demo(demo_observations, history_points=demo_history_points)
demo_results, demo_update_checks = demo_prediction_preview["results"], demo_prediction_preview["update_checks"]
assert all(count > 0 for count in demo_update_checks.values()), demo_update_checks
for case, result in demo_results.items():
    print(f"{case}：成功更新检查 {demo_update_checks[case]}", flush=True)
    display(result.groupby(["status", "reason"]).size().rename("count（采样点数）").rename_axis(
        ["status（预测状态）", "reason（失败原因）"]).to_frame())
display(demo_results[("rv", "HAR")].tail().rename(columns={**VOLATILITY_MODEL_COLUMN_LABELS,
    **demo_prediction_labels}).rename_axis("row（采样行）"))
print("c04 真实小范围逐点试算完成；未写数据文件", flush=True)


c04 rv ARMA 69/69 点；成功更新检查 33


c04 rv ARFIMA 69/69 点；成功更新检查 33


c04 rv HAR 69/69 点；成功更新检查 32


c04 rv MEM 69/69 点；成功更新检查 34


c04 bpv ARMA 69/69 点；成功更新检查 31


...

count（采样点数）
status（预测状态） reason（失败原因）                                  
failed       HAR 设计矩阵非有限或秩不足，具名系数无法识别                     1
             observations 必须为非空有限一维序列；不自动删除缺失行           36
ok                                                       32

('bpv', 'MEM')：成功更新检查 33


,,count（采样点数）
status（预测状态）,reason（失败原因）,
failed,observations 必须为非空有限一维序列；不自动删除缺失行,36
ok,,33


,sample_id（采样编号）,sample_at（预测目标时刻）,trading_date（交易日）,sample_side（采样侧）,training_start_sample_id（训练首个编号）,training_end_sample_id（训练末个编号）,training_points（训练观测数）,model（模型）,status（状态）,reason（原因）,mean_forecast（下一步观测预测）,parameters（具名拟合系数）,update_status（更新状态）,update_reason（更新原因）,updated_mean_forecast（更新后下一步预测）,innovation_variance（观测创新方差）,features（HAR 窗口均值）,components（预测成分）
row（采样行）,,,,,,,,,,,,,,,,,,
64,65,2025-01-02 14:04:19.119334+08:00,2025-01-02,test,35.0,64.0,30,HAR,ok,,0.000002,"{'mean.intercept': 2.0679668994378036e-06, 'mean.window[1]': 0.0, 'mean.window[5]': 0.0, 'mean.window[20]': 0.0}",ok,,0.000002,NaN,"{'window_mean[1]': 1.7059382779054463e-06, 'window_mean[5]': 1.9885256623221324e-06, 'window_mean[20]': 1.95839837522187e-06}","{'intercept': 2.0679668994378036e-06, 'window[1]': 0.0, 'window[5]': 0.0, 'window[20]': 0.0}"
65,66,2025-01-02 14:15:26.888087+08:00,2025-01-02,test,36.0,65.0,30,HAR,ok,,0.000002,"{'mean.intercept': 2.1412499574552004e-06, 'mean.window[1]': 0.0, 'mean.window[5]': 0.0, 'mean.window[20]': 0.0}",ok,,0.000002,NaN,"{'window_mean[1]': 2.7655262332390664e-06, 'window_mean[5]': 2.388496982385577e-06, 'window_mean[20]': 2.0021299416226628e-06}","{'intercept': 2.1412499574552004e-06, 'window[1]': 0.0, 'window[5]': 0.0, 'window[20]': 0.0}"
66,67,2025-01-02 14:24:14.002070+08:00,2025-01-02,test,37.0,66.0,30,HAR,ok,,0.000002,"{'mean.intercept': 2.156966109984832e-06, 'mean.window[1]': 0.0, 'mean.window[5]': 0.06888447676475237, 'mean.window[20]': 0.0}",ok,,0.000002,NaN,"{'window_mean[1]': 3.2426116326479094e-06, 'window_mean[5]': 2.7755775728993352e-06, 'window_mean[20]': 2.0568529157660898e-06}","{'intercept': 2.156966109984832e-06, 'window[1]': 0.0, 'window[5]': 1.9119420882915205e-07, 'window[20]': 0.0}"
67,68,2025-01-02 14:42:16.490935+08:00,2025-01-02,test,38.0,67.0,30,HAR,ok,,0.000002,"{'mean.intercept': 2.2458559461938118e-06, 'mean.window[1]': 0.0, 'mean.window[5]': 0.0, 'mean.window[20]': 0.0}",ok,,0.000002,NaN,"{'window_mean[1]': 1.5864056909291975e-06, 'window_mean[5]': 2.6583722806202916e-06, 'window_mean[20]': 2.0909475849199227e-06}","{'intercept': 2.2458559461938118e-06, 'window[1]': 0.0, 'window[5]': 0.0, 'window[20]': 0.0}"
68,69,2025-01-02 14:58:11.698689+08:00,2025-01-02,test,39.0,68.0,30,HAR,ok,,0.000002,"{'mean.intercept': 2.365167038930596e-06, 'mean.window[1]': 0.0, 'mean.window[5]': 0.0, 'mean.window[20]': 0.0}",ok,,0.000002,NaN,"{'window_mean[1]': 2.8393358046435082e-06, 'window_mean[5]': 2.4279635278730257e-06, 'window_mean[20]': 2.063602813640872e-06}","{'intercept': 2.365167038930596e-06, 'window[1]': 0.0, 'window[5]': 0.0, 'window[20]': 0.0}"


c04 真实小范围逐点试算完成；未写数据文件
